# Trend Analysis: Technology Growth and Decline Trajectories

**User Story 1**: Quantify Technology Growth and Decline Trajectories

This notebook integrates the Modified Mann-Kendall test, Theil-Sen slope estimation, bootstrapping for confidence intervals, and external correlation analysis against GitHub/NPM metrics.

---

### ⚠️ Limitations and Disclosures (FR-011)

**Data Source Limitations**:
- Analysis relies on publicly available Stack Overflow post data. Tags may be applied inconsistently by users.
- External metrics (GitHub stars, NPM downloads) are proxies for adoption but do not capture enterprise usage or private repositories.

**Statistical Limitations**:
- The Modified Mann-Kendall test assumes no significant seasonality after pre-whitening. Residual seasonality may affect p-values.
- Theil-Sen slope estimates are robust to outliers but may be less efficient than parametric estimators if normality assumptions hold.
- Correlation does not imply causation. High correlation with external metrics does not prove that external activity drives Stack Overflow tag usage.

**Scope Limitations**:
- Only tags with ≥12 months of continuous data are included (FR-003).
- Tags classified as "Insufficient Data" (p ≥ 0.05 AND power < 0.8) are reported but excluded from trend significance claims.

---

## 1. Setup and Imports

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project root adjustment for notebook execution
PROJECT_ROOT = Path.cwd().parent
CODE_DIR = PROJECT_ROOT / 'code'
DATA_DIR = PROJECT_ROOT / 'data'

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Import analysis modules
from data.preprocess import load_raw_data, aggregate_monthly_frequencies, filter_min_months, save_processed_data
from analysis.trends import analyze_trends, classify_trend, modified_mann_kendall, theil_sen_slope
from analysis.bootstrapping import run_bootstrapping_analysis, load_processed_data, save_confidence_intervals
from analysis.correlation import run_correlation_analysis, load_trend_data_for_correlation
from utils.state_manager import update_artifact_checksums, load_state
from utils.contract_validation import validate_artifact, load_contract

print(f"Project Root: {PROJECT_ROOT}")
print("Imports successful.")

## 2. Data Loading and Preprocessing

Loads raw data (if available), normalizes tags, aggregates to monthly frequencies, and filters for minimum 12 months of data.

In [ ]:
raw_data_path = DATA_DIR / 'raw' / 'posts_tags.json'
processed_data_path = DATA_DIR / 'processed' / 'monthly_tag_frequencies.json'

if not raw_data_path.exists():
    print(f"Warning: Raw data not found at {raw_data_path}. Attempting to load processed data directly if available.")
    if not processed_data_path.exists():
        raise FileNotFoundError(f"Neither raw data ({raw_data_path}) nor processed data ({processed_data_path}) found. Please run code/data/download.py and code/data/preprocess.py first.")
    else:
        print("Loading existing processed data.")
        processed_df = pd.read_json(processed_data_path)
else:
    print(f"Loading and preprocessing data from {raw_data_path}...")
    # Load raw
    raw_df = load_raw_data(raw_data_path)
    # Normalize and parse
    raw_df['normalized_tags'] = raw_df['tags'].apply(normalize_tags)
    raw_df['date'] = pd.to_datetime(raw_df['creation_date'])
    
    # Aggregate
    monthly_df = aggregate_monthly_frequencies(raw_df)
    
    # Filter
    filtered_df = filter_min_months(monthly_df, min_months=12)
    
    # Save
    save_processed_data(filtered_df, processed_data_path)
    processed_df = filtered_df

print(f"Processed data shape: {processed_df.shape}")
print(processed_df.head())

## 3. Trend Analysis (Modified Mann-Kendall & Theil-Sen)

Performs the core statistical analysis: pre-whitening, Mann-Kendall test, Theil-Sen slope, and classification.

In [ ]:
print("Running trend analysis...")
trend_results = analyze_trends(processed_df)

print(f"Analysis complete. Results for {len(trend_results)} tags.")
print(pd.DataFrame(trend_results).head())

## 4. Bootstrapping for Confidence Intervals

Calculates 95% confidence intervals for Theil-Sen slopes using bootstrapping (FR-010).

In [ ]:
ci_results = run_bootstrapping_analysis(processed_df, n_iterations=1000)
ci_path = DATA_DIR / 'processed' / 'confidence_interval.json'
save_confidence_intervals(ci_results, ci_path)
print(f"Confidence intervals saved to {ci_path}")
print(pd.DataFrame(ci_results).head())

## 5. External Correlation Analysis

Maps tags to GitHub/NPM and computes Pearson correlation with external metrics.

In [ ]:
print("Running correlation analysis...")
correlation_results = run_correlation_analysis(trend_results, processed_df)
print(f"Correlation analysis complete. Mapped {len([r for r in correlation_results if r['status'] == 'mapped'])} tags.")
print(pd.DataFrame(correlation_results).head())

## 6. Aggregation and Final Output

Merges trend, CI, and correlation data into `trend_results.json` and updates state checksums.

In [ ]:
# Merge results
final_results = []
for t in trend_results:
    tag = t['tag']
    ci = next((c for c in ci_results if c['tag'] == tag), None)
    corr = next((c for c in correlation_results if c['tag'] == tag), None)
    
    entry = {
        'tag': tag,
        'slope': t['slope'],
        'p_value': t['p_value'],
        'classification': t['classification'],
        'power': t['power'],
        'mdes': t['mdes'],
        'ci_lower': ci['ci_lower'] if ci else None,
        'ci_upper': ci['ci_upper'] if ci else None,
        'external_source': corr['external_source'] if corr else None,
        'correlation_coefficient': corr['correlation'] if corr else None,
        'correlation_p_value': corr['p_value'] if corr else None
    }
    final_results.append(entry)

output_path = DATA_DIR / 'processed' / 'trend_results.json'
with open(output_path, 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"Final results saved to {output_path}")

# Update State
state_path = PROJECT_ROOT / 'state' / 'projects' / 'PROJ-298-statistical-analysis-of-publicly-availab.yaml'
update_artifact_checksums(state_path, [output_path, ci_path])
print("State file updated.")

## 7. Visualization

Generates summary plots: Classification distribution and Top 10 trends by slope.

In [ ]:
df_results = pd.DataFrame(final_results)

# Plot 1: Classification Distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=df_results, x='classification', order=['Growth', 'Decline', 'Stable', 'Insufficient Data'])
plt.title('Distribution of Tag Trend Classifications')
plt.xlabel('Classification')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(DATA_DIR / 'figures' / 'trend_classification_dist.png')
plt.show()

# Plot 2: Top 10 Trends by Slope (excluding Insufficient Data)
df_valid = df_results[df_results['classification'].isin(['Growth', 'Decline'])].copy()
if not df_valid.empty:
    df_valid = df_valid.sort_values('slope', ascending=False).head(10)
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df_valid, x='slope', y='tag', palette='viridis')
    plt.title('Top 10 Tags by Trend Slope')
    plt.xlabel('Theil-Sen Slope (posts/month)')
    plt.ylabel('Tag')
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'figures' / 'top_trends_slope.png')
    plt.show()
else:
    print("No valid growth/decline tags found for visualization.")

## 8. Summary Statistics

Prints key metrics for the report.

In [ ]:
print("\n=== Trend Analysis Summary ===")
print(f"Total tags analyzed: {len(df_results)}")
print(f"Growth: {len(df_results[df_results['classification'] == 'Growth'])}")
print(f"Decline: {len(df_results[df_results['classification'] == 'Decline'])}")
print(f"Stable: {len(df_results[df_results['classification'] == 'Stable'])}")
print(f"Insufficient Data: {len(df_results[df_results['classification'] == 'Insufficient Data'])}")

if 'correlation_coefficient' in df_results.columns:
    mapped = df_results[df_results['correlation_coefficient'].notna()]
    print(f"\nTags with external correlation data: {len(mapped)}")
    if len(mapped) > 0:
        print(f"Average correlation coefficient: {mapped['correlation_coefficient'].mean():.3f}")

---
### End of Notebook

All outputs have been written to `data/processed/` and `data/figures/`. State file has been updated with checksums.

**Limitations Reminder**: Results are subject to the data quality and statistical assumptions outlined in the header. Correlation does not imply causation.